# Neural Network From Scratch

A simple 2-2-1 neural network (2 inputs, 2 hidden neurons, 1 output neuron), built using only NumPy — no ML frameworks — to understand feedforward and backpropagation (the chain rule) from first principles.

This implementation follows Victor Zhou's neural network tutorial ("Machine Learning for Beginners: An Introduction to Neural Networks"), done to understand backprop and gradient descent from the ground up.

**Task:** predict gender (0 = Male, 1 = Female) from weight and height.

In [17]:
import numpy as np

In [18]:
def sigmoid(x):
    # Activation function: squashes any real number into the range (0, 1)
    # f(x) = 1 / (1 + e^-x)
    return 1 / (1 + np.exp(-x))

def deriv_sigmoid(x):
    # Derivative of sigmoid, needed for backpropagation.
    # Simplifies nicely to f'(x) = f(x) * (1 - f(x)), so we can reuse
    # the already-computed sigmoid output instead of recomputing from scratch.
    fx = sigmoid(x)
    return fx * (1 - fx)

def MSE_loss(y_true, y_pred):
    # Mean Squared Error: average of (actual - predicted)^2 across all samples.
    # Tells us how far off the network's predictions are, on average.
    # y_true and y_pred are numpy arrays of the same length.
    return ((y_true - y_pred) ** 2).mean()

In [22]:
class OurNeuralNetwork:
    '''
    A neural network with:
      - 2 inputs (weight, height)
      - a hidden layer with 2 neurons (h1, h2)
      - an output layer with 1 neuron (o1)

    *** DISCLAIMER ***:
    This code is intentionally simple and educational, NOT optimal.
    Real neural net code (PyTorch/TensorFlow) uses automatic
    differentiation and vectorized matrix ops instead of hand-written
    gradients like this. This is meant purely to trace, line by line,
    how one small network computes forward and backward passes.
    '''
    def __init__(self):
        # Weights — randomly initialized from a standard normal distribution
        # (mean 0, std 1). Random init breaks symmetry: if all weights
        # started identical, every neuron would learn the same thing.
        self.w1 = np.random.normal()
        self.w2 = np.random.normal()
        self.w3 = np.random.normal()
        self.w4 = np.random.normal()
        self.w5 = np.random.normal()
        self.w6 = np.random.normal()

        # Biases — also randomly initialized, one per neuron (3 neurons
        # total: h1, h2, o1), so each neuron can shift its own threshold
        # independently of the others.
        self.b1 = np.random.normal()
        self.b2 = np.random.normal()
        self.b3 = np.random.normal()

    def feedforward(self, x):
        # x is a numpy array with 2 elements: [weight, height] (already shifted).
        # Each neuron: weighted sum of inputs + bias, then squashed through sigmoid.
        h1 = sigmoid(self.w1 * x[0] + self.w2 * x[1] + self.b1)
        h2 = sigmoid(self.w3 * x[0] + self.w4 * x[1] + self.b2)
        # h1 and h2 (already-activated values) become the inputs to the output neuron.
        o1 = sigmoid(self.w5 * h1 + self.w6 * h2 + self.b3)
        return o1

    def train(self, data, all_y_trues):
        '''
        - data is a (n x 2) numpy array, n = # of samples in the dataset.
        - all_y_trues is a numpy array with n elements — the actual,
          known-correct labels (ground truth), not predictions.
          Elements in all_y_trues correspond to those in data.
        '''

        learn_rate = 0.1   # step size for each gradient descent update
        epochs = 1000      # number of times to loop through the entire dataset

        for epoch in range(epochs):
            # Stochastic gradient descent: update weights/biases after
            # every single training example, not just once per epoch.
            for x, y_true in zip(data, all_y_trues):

                # --- Feedforward pass ---
                # Keep the pre-activation sums (sum_h1, sum_h2, sum_o1) around
                # because we'll need them again for the derivative calculations below.
                sum_h1 = self.w1 * x[0] + self.w2 * x[1] + self.b1
                h1 = sigmoid(sum_h1)

                sum_h2 = self.w3 * x[0] + self.w4 * x[1] + self.b2
                h2 = sigmoid(sum_h2)

                sum_o1 = self.w5 * h1 + self.w6 * h2 + self.b3
                o1 = sigmoid(sum_o1)
                y_pred = o1

                # --- Backpropagation: calculate partial derivatives ---
                # Naming convention: d_L_d_w1 means "partial derivative of
                # Loss L with respect to w1".

                # dL/dy_pred — since L = (1 - y_pred)^2 (for y_true = 1 case,
                # generalized here as -2*(y_true - y_pred))
                d_L_d_ypred = -2 * (y_true - y_pred)

                # --- Neuron o1 (output layer) ---
                # How y_pred changes w.r.t. each weight/bias feeding into o1.
                # Each of these = (the thing multiplying that weight) * f'(sum_o1)
                d_ypred_d_w5 = h1 * deriv_sigmoid(sum_o1)
                d_ypred_d_w6 = h2 * deriv_sigmoid(sum_o1)
                # For the bias, nothing "multiplies" it (it's added on its own),
                # so its local derivative is just f'(sum_o1) * 1.
                d_ypred_d_b3 = deriv_sigmoid(sum_o1)

                # How y_pred changes w.r.t. h1 and h2 — needed to keep propagating
                # the gradient backward into the hidden layer.
                d_ypred_d_h1 = self.w5 * deriv_sigmoid(sum_o1)
                d_ypred_d_h2 = self.w6 * deriv_sigmoid(sum_o1)

                # --- Neuron h1 (hidden layer) ---
                # How h1 changes w.r.t. its own weights/bias.
                d_h1_d_w1 = x[0] * deriv_sigmoid(sum_h1)
                d_h1_d_w2 = x[1] * deriv_sigmoid(sum_h1)
                d_h1_d_b1 = deriv_sigmoid(sum_h1)

                # --- Neuron h2 (hidden layer) ---
                # How h2 changes w.r.t. its own weights/bias.
                d_h2_d_w3 = x[0] * deriv_sigmoid(sum_h2)
                d_h2_d_w4 = x[1] * deriv_sigmoid(sum_h2)
                d_h2_d_b2 = deriv_sigmoid(sum_h2)

                # --- Update weights and biases (gradient descent step) ---
                # new_value = old_value - learn_rate * (chain rule product of
                # all the partial derivatives linking this parameter to the loss)

                # Neuron h1's weights/bias: loss -> y_pred -> h1 -> w1/w2/b1
                self.w1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w1
                self.w2 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_w2
                self.b1 -= learn_rate * d_L_d_ypred * d_ypred_d_h1 * d_h1_d_b1

                # Neuron h2's weights/bias: loss -> y_pred -> h2 -> w3/w4/b2
                self.w3 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w3
                self.w4 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_w4
                self.b2 -= learn_rate * d_L_d_ypred * d_ypred_d_h2 * d_h2_d_b2

                # Neuron o1's weights/bias: loss -> y_pred -> w5/w6/b3 directly
                self.w5 -= learn_rate * d_L_d_ypred * d_ypred_d_w5
                self.w6 -= learn_rate * d_L_d_ypred * d_ypred_d_w6
                self.b3 -= learn_rate * d_L_d_ypred * d_ypred_d_b3

            # --- Calculate and print total loss at the end of every 10th epoch ---
            # (Just for monitoring progress — not part of the training math itself.)
            if epoch % 10 == 0:
                y_preds = np.apply_along_axis(self.feedforward, 1, data)
                loss = MSE_loss(all_y_trues, y_preds)
                print("Epoch %d loss: %.3f" % (epoch, loss))

# --- Define dataset ---
# Weight and height have been shifted (weight - 135, height - 66) so the
# values are small and centered around 0. This keeps the weighted sums fed
# into sigmoid within its steep, sensitive region instead of the flat,
# saturated extremes where gradients vanish and learning stalls.
data = np.array([
    [-2, -1],  # Alice
    [25, 6],   # Bob
    [17, 4],   # Charlie
    [-15, -6], # Diana
])

# Ground-truth labels: 1 = Female, 0 = Male (matches order of rows above)
all_y_trues = np.array([
    1,  # Alice
    0,  # Bob
    0,  # Charlie
    1,  # Diana
])

# --- Train our neural network! ---
network = OurNeuralNetwork()
network.train(data, all_y_trues)

# --- Make some predictions on new, unseen data ---
# Inputs must be shifted the same way as the training data (weight - 135, height - 66).
emily = np.array([-7, -3])  # 128 pounds, 63 inches -> expect close to 1 (F)
frank = np.array([20, 2])   # 155 pounds, 68 inches -> expect close to 0 (M)
print("Emily: %.3f" % network.feedforward(emily))  # ~0.97 -> F
print("Frank: %.3f" % network.feedforward(frank))  # ~0.04 -> M

Epoch 0 loss: 0.291
Epoch 10 loss: 0.233
Epoch 20 loss: 0.192
Epoch 30 loss: 0.161
Epoch 40 loss: 0.135
Epoch 50 loss: 0.111
Epoch 60 loss: 0.090
Epoch 70 loss: 0.071
Epoch 80 loss: 0.056
Epoch 90 loss: 0.045
Epoch 100 loss: 0.037
Epoch 110 loss: 0.031
Epoch 120 loss: 0.026
Epoch 130 loss: 0.023
Epoch 140 loss: 0.020
Epoch 150 loss: 0.018
Epoch 160 loss: 0.016
Epoch 170 loss: 0.015
Epoch 180 loss: 0.013
Epoch 190 loss: 0.012
Epoch 200 loss: 0.011
Epoch 210 loss: 0.011
Epoch 220 loss: 0.010
Epoch 230 loss: 0.009
Epoch 240 loss: 0.009
Epoch 250 loss: 0.008
Epoch 260 loss: 0.008
Epoch 270 loss: 0.007
Epoch 280 loss: 0.007
Epoch 290 loss: 0.007
Epoch 300 loss: 0.006
Epoch 310 loss: 0.006
Epoch 320 loss: 0.006
Epoch 330 loss: 0.006
Epoch 340 loss: 0.005
Epoch 350 loss: 0.005
Epoch 360 loss: 0.005
Epoch 370 loss: 0.005
Epoch 380 loss: 0.005
Epoch 390 loss: 0.005
Epoch 400 loss: 0.004
Epoch 410 loss: 0.004
Epoch 420 loss: 0.004
Epoch 430 loss: 0.004
Epoch 440 loss: 0.004
Epoch 450 loss: 0.004